# ExtremeTrack UniTrack Trainer (Kaggle)

This notebook trains and validates a **UniTrack-style SOT model** on the same ExtremeTrack dataset used by `vistac.ipynb`.

What is taken from UniTrack:
- the **appearance model idea** from the repo
- the **modified ResNet-18 backbone** pattern
- the **SiamFC correlation head / tracking formulation** used by UniTrack for SOT

Why this notebook is custom:
- the upstream UniTrack repo is mainly an **evaluation framework** for pre-trained appearance models
- it does **not** provide a direct Kaggle training pipeline for the ExtremeTrack dataset
- this notebook adds an ExtremeTrack training loop, Kaggle data loading, checkpointing, and QP validation

Repo referenced by request:
- [UniTrack](https://github.com/Zhongdao/UniTrack)

In [ ]:
!pip -q install brisque opencv-python-headless

In [ ]:
import os
import gc
import json
import math
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List

import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision
import torchvision.transforms.functional as TF

from brisque import BRISQUE

# ------------------------------
# Runtime + Kaggle path config
# ------------------------------
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
LOCAL_DATASET_ROOT = Path('./ExtremeTrack')
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable GPU runtime first.')


def find_dataset_root() -> Path:
    candidates = [
        KAGGLE_INPUT_ROOT,
        Path('/kaggle/working'),
        LOCAL_DATASET_ROOT,
        Path('.'),
    ]
    required = ('ExtremeTrack_train.json', 'ExtremeTrack_val.json')

    for c in candidates:
        if all((c / name).exists() for name in required):
            return c

    for base in candidates:
        if not base.exists():
            continue
        try:
            for p in base.rglob('ExtremeTrack_train.json'):
                root = p.parent
                if (root / 'ExtremeTrack_val.json').exists():
                    return root
        except Exception:
            continue

    raise FileNotFoundError(
        'ExtremeTrack_train.json and ExtremeTrack_val.json were not found. '
        'In Kaggle, add the ExtremeTrack dataset from the Input sidebar first.'
    )


DATASET_ROOT = find_dataset_root()
OUTPUT_ROOT = Path('/kaggle/working/unitrack_extremetrack_outputs') if IN_KAGGLE else Path('./unitrack_extremetrack_outputs')
for subdir in ['checkpoints', 'predictions', 'experiments', 'cache']:
    (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print('DATASET_ROOT =', DATASET_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)

In [ ]:
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json(path: Path, payload: dict):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)


@dataclass
class SequenceRecord:
    name: str
    video_dir: str
    image_paths: List[str]
    gt_rect: List[List[float]]
    condition: str


def load_sequences(annotation_name: str, condition_filter='all'):
    obj = load_json(DATASET_ROOT / annotation_name)
    seqs = []
    for name, seq in obj.items():
        condition = 'haze' if name.lower().endswith('haze') else 'rain'
        if condition_filter != 'all' and condition != condition_filter:
            continue
        seqs.append(SequenceRecord(
            name=name,
            video_dir=seq['video_dir'],
            image_paths=seq['img_names'],
            gt_rect=seq['gt_rect'],
            condition=condition,
        ))
    return seqs


def clip_box_xywh(box, width, height, min_size=2.0):
    x, y, w, h = box
    w = max(min_size, min(float(w), max(min_size, width - 1)))
    h = max(min_size, min(float(h), max(min_size, height - 1)))
    x = min(max(0.0, float(x)), max(0.0, width - w))
    y = min(max(0.0, float(y)), max(0.0, height - h))
    return [x, y, w, h]


def image_from_path(rel_path: str) -> Image.Image:
    return Image.open(DATASET_ROOT / rel_path).convert('RGB')


def crop_resize_rect(image: Image.Image, cx: float, cy: float, crop_w: float, crop_h: float, out_size: int):
    arr = np.asarray(image)
    h, w = arr.shape[:2]

    left = cx - crop_w / 2.0
    top = cy - crop_h / 2.0
    right = cx + crop_w / 2.0
    bottom = cy + crop_h / 2.0

    pad_left = max(0, int(math.ceil(-left)))
    pad_top = max(0, int(math.ceil(-top)))
    pad_right = max(0, int(math.ceil(right - w)))
    pad_bottom = max(0, int(math.ceil(bottom - h)))

    if pad_left or pad_top or pad_right or pad_bottom:
        arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode='edge')
        left += pad_left
        top += pad_top
        right += pad_left
        bottom += pad_top

    x1 = int(round(left))
    y1 = int(round(top))
    x2 = int(round(right))
    y2 = int(round(bottom))
    x2 = max(x1 + 2, x2)
    y2 = max(y1 + 2, y2)

    crop = arr[y1:y2, x1:x2]
    crop_img = Image.fromarray(crop).resize((out_size, out_size), Image.Resampling.BILINEAR)
    info = {
        'left': float(x1 - pad_left),
        'top': float(y1 - pad_top),
        'crop_w': float(x2 - x1),
        'crop_h': float(y2 - y1),
        'out_size': int(out_size),
    }
    return crop_img, info


def crop_box_to_response_center(box, crop_info, response_size):
    x, y, w, h = box
    cx = x + w / 2.0
    cy = y + h / 2.0

    cx_n = (cx - crop_info['left']) / crop_info['crop_w']
    cy_n = (cy - crop_info['top']) / crop_info['crop_h']

    cx_r = cx_n * (response_size - 1)
    cy_r = cy_n * (response_size - 1)
    return float(cx_r), float(cy_r)


def response_offset_to_image(dx, dy, crop_w, crop_h, response_size):
    return float(dx * crop_w / response_size), float(dy * crop_h / response_size)


def restore_frame(image: Image.Image, condition: str):
    bgr = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8, 8)).apply(l)
    merged = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)
    if condition == 'rain':
        enhanced = cv2.medianBlur(enhanced, 3)
    else:
        enhanced = cv2.bilateralFilter(enhanced, 5, 35, 35)
    return Image.fromarray(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))


def bbox_iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh
    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter + 1e-6
    return float(inter / union)


def center_error(a, b):
    acx = a[0] + a[2] / 2.0
    acy = a[1] + a[3] / 2.0
    bcx = b[0] + b[2] / 2.0
    bcy = b[1] + b[3] / 2.0
    return float(((acx - bcx) ** 2 + (acy - bcy) ** 2) ** 0.5)

In [ ]:
# ---------------------------------------------------------
# UniTrack-style model pieces adapted from the GitHub repo:
# - modified ResNet-18 appearance backbone
# - SiamFC correlation head
# ---------------------------------------------------------

class UniTrackResNet18Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
        base = torchvision.models.resnet18(weights=weights)

        # Same idea as UniTrack's repo: reduce output stride by removing layer4
        # and setting stride in later stages to 1.
        for layer in [base.layer3, base.layer4]:
            for m in layer.modules():
                if isinstance(m, nn.Conv2d) and m.stride == (2, 2):
                    m.stride = (1, 1)
                if isinstance(m, nn.MaxPool2d) and m.stride == 2:
                    m.stride = 1

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x


class AppearanceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = UniTrackResNet18Backbone()

    def forward(self, x):
        return self.model(x)


class CorrUp(nn.Module):
    def __init__(self):
        super().__init__()

    def conv2d_group(self, x, kernel):
        batch = x.size(0)
        pk = kernel.view(-1, x.size(1), kernel.size(2), kernel.size(3))
        px = x.view(1, -1, x.size(2), x.size(3))
        po = F.conv2d(px, pk, groups=batch)
        po = po.view(batch, -1, po.size(2), po.size(3))
        return po

    def forward(self, zf, xf):
        return 0.1 * self.conv2d_group(xf, zf)


class UniTrackSiamFC(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = AppearanceModel()
        self.corr = CorrUp()

    def extract_template_kernel(self, z):
        zf = self.features(z)
        _, _, ts_h, ts_w = zf.shape
        bg_h = ts_h // 2 - int(ts_h // (2 * (3.5 + 1)))
        ed_h = ts_h // 2 + int(ts_h // (2 * (3.5 + 1)))
        bg_w = ts_w // 2 - int(ts_w // (2 * (3.5 + 1)))
        ed_w = ts_w // 2 + int(ts_w // (2 * (3.5 + 1)))
        return zf[:, :, bg_h:ed_h, bg_w:ed_w]

    def response(self, template, search, search_window=None):
        zf = self.extract_template_kernel(template)
        xf = self.features(search)
        if search_window is not None:
            xf = xf * search_window
        return self.corr(zf, xf)


def make_hann_window(shape, device):
    h, w = int(shape[-2]), int(shape[-1])
    window = np.outer(np.hanning(h), np.hanning(w)).astype(np.float32)
    return torch.from_numpy(window).to(device).view(1, 1, h, w)


@torch.no_grad()
def infer_response_size(model, template_size, search_size, device):
    model.eval()
    z = torch.zeros(1, 3, template_size, template_size, device=device)
    x = torch.zeros(1, 3, search_size, search_size, device=device)
    resp = model.response(z, x)
    return int(resp.shape[-1])

In [ ]:
CFG = {
    'condition': 'all',
    'epochs': 6,
    'batch_size': 16,
    'num_workers': 4,
    'samples_per_epoch': 6000,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'template_size': 256,
    'search_size': 320,
    'template_padding': 1.5,
    'search_padding': 3.5,
    'max_frame_gap': 16,
    'use_restoration': False,
    'use_amp': True,
    'channels_last': True,
    'compile_model': False,
    'max_train_batches': 0,
    'max_val_sequences': 0,
}

model = UniTrackSiamFC().to(DEVICE)
if CFG['channels_last']:
    model = model.to(memory_format=torch.channels_last)
if CFG['compile_model'] and hasattr(torch, 'compile'):
    model = torch.compile(model, mode='reduce-overhead')

RESPONSE_SIZE = infer_response_size(model, CFG['template_size'], CFG['search_size'], DEVICE)
print('RESPONSE_SIZE =', RESPONSE_SIZE)


class ExtremeTrackPairDataset(Dataset):
    def __init__(self, annotation_name='ExtremeTrack_train.json', condition_filter='all', samples_per_epoch=4000):
        self.records = load_sequences(annotation_name, condition_filter)
        self.samples_per_epoch = samples_per_epoch

    def __len__(self):
        return self.samples_per_epoch

    def make_response_target(self, cx_r, cy_r):
        yy, xx = np.mgrid[0:RESPONSE_SIZE, 0:RESPONSE_SIZE].astype(np.float32)
        dist2 = (xx - cx_r) ** 2 + (yy - cy_r) ** 2
        pos = np.exp(-dist2 / (2.0 * (RESPONSE_SIZE / 10.0) ** 2)).astype(np.float32)
        return torch.from_numpy(pos).unsqueeze(0)

    def __getitem__(self, idx):
        rec = random.choice(self.records)
        end_idx = len(rec.image_paths) - 1
        s_idx = random.randint(1, end_idx)
        t_idx = random.randint(max(0, s_idx - CFG['max_frame_gap']), s_idx - 1)

        t_img = image_from_path(rec.image_paths[t_idx])
        s_img = image_from_path(rec.image_paths[s_idx])
        t_box = [float(v) for v in rec.gt_rect[t_idx]]
        s_box = [float(v) for v in rec.gt_rect[s_idx]]

        tcx = t_box[0] + t_box[2] / 2.0
        tcy = t_box[1] + t_box[3] / 2.0
        t_crop_w = t_box[2] * (1.0 + CFG['template_padding'])
        t_crop_h = t_box[3] * (1.0 + CFG['template_padding'])

        scx = t_box[0] + t_box[2] / 2.0
        scy = t_box[1] + t_box[3] / 2.0
        s_crop_w = t_box[2] * (1.0 + CFG['search_padding'])
        s_crop_h = t_box[3] * (1.0 + CFG['search_padding'])

        template_crop, _ = crop_resize_rect(t_img, tcx, tcy, t_crop_w, t_crop_h, CFG['template_size'])
        search_crop, search_info = crop_resize_rect(s_img, scx, scy, s_crop_w, s_crop_h, CFG['search_size'])

        if CFG['use_restoration']:
            template_crop = restore_frame(template_crop, rec.condition)
            search_crop = restore_frame(search_crop, rec.condition)

        cx_r, cy_r = crop_box_to_response_center(s_box, search_info, RESPONSE_SIZE)
        target_map = self.make_response_target(cx_r, cy_r)

        return {
            'template': TF.to_tensor(template_crop),
            'search': TF.to_tensor(search_crop),
            'target_map': target_map,
        }


train_ds = ExtremeTrackPairDataset(
    'ExtremeTrack_train.json',
    condition_filter=CFG['condition'],
    samples_per_epoch=CFG['samples_per_epoch'],
)
val_seqs = load_sequences('ExtremeTrack_val.json', condition_filter=CFG['condition'])
if CFG['max_val_sequences'] > 0:
    val_seqs = val_seqs[:CFG['max_val_sequences']]

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=True,
    persistent_workers=CFG['num_workers'] > 0,
    prefetch_factor=4 if CFG['num_workers'] > 0 else None,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scaler = GradScaler(device=DEVICE.type, enabled=CFG['use_amp'])


def response_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target)


def train_one_epoch(epoch):
    model.train()
    losses = []
    iterator = train_loader
    if CFG['max_train_batches'] > 0:
        iterator = []
        for i, batch in enumerate(train_loader):
            iterator.append(batch)
            if i + 1 >= CFG['max_train_batches']:
                break

    for batch in tqdm(iterator, desc=f'train epoch {epoch}'):
        template = batch['template'].to(DEVICE, non_blocking=True)
        search = batch['search'].to(DEVICE, non_blocking=True)
        target = batch['target_map'].to(DEVICE, non_blocking=True)

        if CFG['channels_last']:
            template = template.contiguous(memory_format=torch.channels_last)
            search = search.contiguous(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=DEVICE.type, enabled=CFG['use_amp']):
            logits = model.response(template, search)
            loss = response_loss(logits, target)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.item()))

    return float(np.mean(losses)) if losses else 0.0

In [ ]:
@torch.no_grad()
def track_sequence(seq: SequenceRecord):
    model.eval()

    img0 = image_from_path(seq.image_paths[0])
    cur = [float(v) for v in seq.gt_rect[0]]

    tcx = cur[0] + cur[2] / 2.0
    tcy = cur[1] + cur[3] / 2.0
    t_crop_w = cur[2] * (1.0 + CFG['template_padding'])
    t_crop_h = cur[3] * (1.0 + CFG['template_padding'])
    template_crop, _ = crop_resize_rect(img0, tcx, tcy, t_crop_w, t_crop_h, CFG['template_size'])
    if CFG['use_restoration']:
        template_crop = restore_frame(template_crop, seq.condition)

    template = TF.to_tensor(template_crop).unsqueeze(0).to(DEVICE)
    if CFG['channels_last']:
        template = template.contiguous(memory_format=torch.channels_last)

    scale_factors = [1.0 / 1.03, 1.0, 1.03]
    pred_boxes = [cur.copy()]
    confs = [1.0]

    for fi in range(1, len(seq.image_paths)):
        img = image_from_path(seq.image_paths[fi])
        best_score = -1e9
        best_box = cur.copy()

        for scale in scale_factors:
            cx = cur[0] + cur[2] / 2.0
            cy = cur[1] + cur[3] / 2.0
            crop_w = cur[2] * (1.0 + CFG['search_padding']) * scale
            crop_h = cur[3] * (1.0 + CFG['search_padding']) * scale
            search_crop, search_info = crop_resize_rect(img, cx, cy, crop_w, crop_h, CFG['search_size'])
            if CFG['use_restoration']:
                search_crop = restore_frame(search_crop, seq.condition)

            search = TF.to_tensor(search_crop).unsqueeze(0).to(DEVICE)
            if CFG['channels_last']:
                search = search.contiguous(memory_format=torch.channels_last)

            logits = model.response(template, search)
            score_map = torch.sigmoid(logits)[0, 0]
            max_score = float(score_map.max().item())
            max_idx = int(score_map.argmax().item())
            row = max_idx // score_map.shape[1]
            col = max_idx % score_map.shape[1]
            dx = col - (score_map.shape[1] - 1) / 2.0
            dy = row - (score_map.shape[0] - 1) / 2.0
            shift_x, shift_y = response_offset_to_image(dx, dy, search_info['crop_w'], search_info['crop_h'], score_map.shape[1])

            pred_cx = cx + shift_x
            pred_cy = cy + shift_y
            pred_w = cur[2] * scale
            pred_h = cur[3] * scale
            pred_box = clip_box_xywh(
                [pred_cx - pred_w / 2.0, pred_cy - pred_h / 2.0, pred_w, pred_h],
                width=img.size[0],
                height=img.size[1],
            )

            if max_score > best_score:
                best_score = max_score
                best_box = pred_box

        cur = best_box
        pred_boxes.append(cur.copy())
        confs.append(best_score)

    return pred_boxes, confs


def validate_qp(iqa_cache_path=OUTPUT_ROOT / 'cache' / 'val_iqa_cache.json'):
    br = BRISQUE(url=False)
    cache = {}
    if iqa_cache_path.exists():
        try:
            cache = load_json(iqa_cache_path)
        except Exception:
            cache = {}

    def get_iqa(path_str):
        if path_str in cache:
            return float(cache[path_str])
        arr = np.asarray(Image.open(path_str).convert('RGB'))
        b = float(br.score(arr))
        v = float(np.clip(1.0 - b / 100.0, 0.0, 1.0))
        cache[path_str] = v
        return v

    metrics = []
    preds_payload = {}

    for seq in tqdm(val_seqs, desc='validate'):
        pred_boxes, confs = track_sequence(seq)
        gt = [[float(v) for v in b] for b in seq.gt_rect]
        ious = np.array([bbox_iou(p, g) for p, g in zip(pred_boxes, gt)], dtype=np.float32)
        ces = np.array([center_error(p, g) for p, g in zip(pred_boxes, gt)], dtype=np.float32)
        iqas = np.array([get_iqa(str(DATASET_ROOT / p)) for p in seq.image_paths], dtype=np.float32)

        precision_20 = float((ces <= 20.0).mean())
        thresholds = np.linspace(0.0, 1.0, 21, dtype=np.float32)
        success_curve = np.array([(ious >= t).mean() for t in thresholds], dtype=np.float32)
        success_auc = float(success_curve.mean())
        qp = float(((iqas * ces) < 15.0).mean())

        metrics.append({
            'mean_iou': float(ious.mean()),
            'precision_20': precision_20,
            'success_auc': success_auc,
            'mean_iqa': float(iqas.mean()),
            'qp': qp,
        })

        preds_payload[seq.video_dir] = {
            'pred_rect': [[round(v, 3) for v in b] for b in pred_boxes],
            'confidence': [round(v, 5) for v in confs],
        }

    save_json(iqa_cache_path, cache)
    agg = {k: float(np.mean([m[k] for m in metrics])) for k in metrics[0].keys()}
    return agg, preds_payload


best_qp = -1.0
best_ckpt = None
history = []

for ep in range(1, CFG['epochs'] + 1):
    t0 = time.time()
    train_loss = train_one_epoch(ep)
    val_metrics, val_preds = validate_qp()
    elapsed = time.time() - t0

    ckpt_name = f"unitrack_siamfc_epoch{ep:02d}_iou{val_metrics['mean_iou']:.4f}_qp{val_metrics['qp']:.4f}.pt"
    ckpt_path = OUTPUT_ROOT / 'checkpoints' / ckpt_name
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'epoch': ep,
        'metrics': val_metrics,
        'cfg': CFG,
        'response_size': RESPONSE_SIZE,
    }, ckpt_path)

    pred_path = OUTPUT_ROOT / 'predictions' / f'unitrack_siamfc_epoch{ep:02d}_val_predictions.json'
    save_json(pred_path, val_preds)

    row = {
        'epoch': ep,
        'train_loss': float(train_loss),
        **val_metrics,
        'checkpoint': str(ckpt_path),
        'prediction_file': str(pred_path),
        'epoch_seconds': round(elapsed, 2),
    }
    history.append(row)
    print(row)

    if val_metrics['qp'] > best_qp:
        best_qp = val_metrics['qp']
        best_ckpt = ckpt_path

    gc.collect()
    torch.cuda.empty_cache()

summary = {
    'best_qp': best_qp,
    'best_checkpoint': str(best_ckpt) if best_ckpt else '',
    'epochs': CFG['epochs'],
    'condition': CFG['condition'],
    'dataset_root': str(DATASET_ROOT),
    'response_size': RESPONSE_SIZE,
}
save_json(OUTPUT_ROOT / 'experiments' / 'unitrack_siamfc_history.json', {'history': history, 'summary': summary})
save_json(OUTPUT_ROOT / 'leaderboard.json', {'unitrack_siamfc': summary})

print('
Done')
print('Summary:', summary)
print('Output folder:', OUTPUT_ROOT)